# NovaOps — The Same PharmaOps Curriculum, Rebuilt on Google ADK

Same fictional company concept, same drug-batch journey, same 11 chapters —
this time using **Google's Agent Development Kit (ADK)** instead of
LangGraph. Company name: **NovaMed Inc.**

## Why the code looks different from the LangGraph version

ADK's mental model is different on purpose, and this notebook leans into
that rather than hiding it:

- **No manual graph wiring for simple agents.** An `LlmAgent` with `tools=[...]`
  already runs its own reason -> call tool -> observe -> reason loop
  internally. There's no `ToolNode` or `tools_condition` to wire — that's
  Example 3 below, and it's dramatically shorter than the LangGraph version.
- **Multi-agent delegation is automatic, not manually routed.** In the
  LangGraph guide, the Manufacturing Supervisor used a Pydantic
  `RoutingDecision` model and explicit conditional edges. In ADK, a
  coordinator `LlmAgent` with `sub_agents=[...]` gets delegation handled by
  the framework itself — you just describe each specialist, and the
  coordinator's LLM decides who to hand off to.
- **Sequential pipelines use `SequentialAgent`**, and pass data between
  steps via **session state templating** (`{key}` in an instruction string)
  instead of a `TypedDict` state schema.
- **Reflection loops use `LoopAgent`** plus an `exit_loop` tool the reviewer
  calls when satisfied — not a manually-written conditional edge that
  checks for the word "APPROVED".
- **Example 1 stays non-agentic on purpose.** Just like in the LangGraph
  version, the Batch Stage Tracker is plain Python — no LLM, no ADK. It's
  the same teaching point either framework: state moving through steps
  isn't an agent by itself.

## The Drug Manufacturing Journey — where each agent fits

```
RAW MATERIAL INTAKE  ->  FORMULATION  ->  QUALITY CONTROL
Ch.1 Batch Tracker        Ch.1 Batch        Ch.3 Batch Release Agent
Ch.7 Inventory Agent      Tracker           (LlmAgent + tools)
                                                  |
                                      +-----------+-----------+
                                    PASS                    FAIL
                                      |                       |
                          REGULATORY REVIEW           DEVIATION HANDLING
                          Ch.4 Coordinator ->          Ch.8 LoopAgent
                          regulatory_agent              (draft -> review -> revise)
                                      |                       |
                          BATCH DOCUMENTATION          back to FORMULATION
                          Ch.4 Coordinator ->
                          documentation_agent
                                      |
                          HUMAN PHARMACIST SIGN-OFF    <- agents inform,
                                      |                    human decides
                          MARKET-FACING DRUG INFO
                          Ch.6 Drug Info & Pricing Agent
                                      |
                          POST-MARKET SURVEILLANCE
                          Ch.10 Adverse Event Screener (LoopAgent)
                                      |
                          ONGOING COMPLIANCE
                          Ch.11 Regulatory Guidance Summarizer

R&D (Ch.5, SequentialAgent) feeds new compounds INTO raw material intake.
SOP Assistant (Ch.2) and Plant Operations Router (Ch.9) run alongside
every stage above, supporting plant-floor operators directly.
```

## How to run this notebook

1. Run **Setup** to install `google-adk`.
2. Run **Config** and choose a model provider.
   - `google` needs a `GOOGLE_API_KEY` (free tier available at
     [aistudio.google.com/apikey](https://aistudio.google.com/apikey)).
   - `ollama` runs free and local via ADK's LiteLLM integration — install
     [Ollama](https://ollama.com/download), then `ollama pull llama3.2`,
     before running. (Requires `litellm` to be installed alongside ADK.)
3. Run cells top to bottom — later chapters reuse agents built earlier,
   exactly like the LangGraph notebook.


## Setup

In [ ]:
%pip install -q google-adk litellm pandas pydantic python-dotenv
print("Dependencies installed.")

## Config — choose your model provider

`get_model()` is used everywhere below, so this is the only place you
need to change providers.

In [ ]:
import os

MODEL_PROVIDER = "google"   # "google" or "ollama"
GOOGLE_MODEL = "gemini-2.5-flash"
OLLAMA_MODEL = "llama3.2"

# If using google: set your key (or export GOOGLE_API_KEY before starting Jupyter)
# os.environ["GOOGLE_API_KEY"] = "your-key-here"

def get_model():
    """Returns the model identifier/wrapper configured by MODEL_PROVIDER above."""
    if MODEL_PROVIDER == "ollama":
        from google.adk.models.lite_llm import LiteLlm
        return LiteLlm(model=f"ollama_chat/{OLLAMA_MODEL}")
    return GOOGLE_MODEL

print(f"Provider: {MODEL_PROVIDER}")

In [ ]:
import asyncio
from google.genai import types
from google.adk.sessions import InMemorySessionService

APP_NAME = "novamed_ops"
novamed_session_service = InMemorySessionService()

async def call_agent(runner, user_id: str, session_id: str, query: str) -> str:
    """
    Shared helper used by every example below: sends one message to an ADK
    Runner and returns the final text response.
    """
    content = types.Content(role="user", parts=[types.Part(text=query)])
    final_text = ""
    async for event in runner.run_async(user_id=user_id, session_id=session_id, new_message=content):
        if event.is_final_response() and event.content and event.content.parts:
            final_text = event.content.parts[0].text or final_text
    return final_text

async def new_session(user_id: str, session_id: str, initial_state: dict = None):
    """Creates a fresh in-memory session for a given user/session id pair."""
    return await novamed_session_service.create_session(
        app_name=APP_NAME, user_id=user_id, session_id=session_id,
        state=initial_state or {},
    )

---
# Example 1 — Batch Stage Tracker (not an agent)

**Use case:** Before NovaOps can make any decision about a batch, it needs
a record of what's happened to that batch so far. This step is deliberately
plain Python — **no LLM, no ADK agent** — because there's no reasoning or
decision to make, just deterministic state transformation. Just like the
LangGraph version of this chapter, this is the non-agentic baseline every
later example gets compared against.

In [ ]:
def log_material_receipt(batch_state: dict) -> dict:
    """Record that raw materials for this batch have arrived and been logged."""
    batch_state["stage_log"] += " -> Raw materials received"
    print(f"[MATERIAL RECEIPT] {batch_state['stage_log']}")
    return batch_state

def log_formulation_complete(batch_state: dict) -> dict:
    """Record that the batch has been formulated per the master batch record."""
    batch_state["stage_log"] += " -> Formulation complete"
    print(f"[FORMULATION] {batch_state['stage_log']}")
    return batch_state

def log_qc_check(batch_state: dict) -> dict:
    """Record that the batch has passed through the quality control checkpoint."""
    batch_state["stage_log"] += " -> QC checkpoint passed"
    print(f"[QUALITY CONTROL] {batch_state['stage_log']}")
    return batch_state

batch_state = {"batch_id": "NM-2026-0091", "stage_log": "Batch opened"}
batch_state = log_material_receipt(batch_state)
batch_state = log_formulation_complete(batch_state)
batch_state = log_qc_check(batch_state)
print()
print(batch_state)

---
# Example 2 — SOP Assistant

**Use case:** Plant operators constantly need to check Standard Operating
Procedures without pulling a supervisor off the floor. This is the first
real ADK agent in NovaOps: an `LlmAgent` with no tools, just a persona and
a session so it remembers earlier turns in the same conversation.

In [ ]:
from google.adk.agents import Agent
from google.adk.runners import Runner

sop_assistant_agent = Agent(
    name="sop_assistant_agent",
    model=get_model(),
    description="Answers plant operator questions about NovaMed SOPs.",
    instruction="""You are the NovaMed SOP Assistant. You help plant operators
    understand standard operating procedures for batch handling, cleanroom
    protocol, and GMP documentation requirements. Be precise, and say so if a
    procedure isn't in your knowledge rather than guessing.""",
)

sop_assistant_runner = Runner(
    agent=sop_assistant_agent,
    app_name=APP_NAME,
    session_service=novamed_session_service,
)

await new_session(user_id="operator-1", session_id="sop-session-1")

answer_1 = await call_agent(sop_assistant_runner, "operator-1", "sop-session-1",
    "What's the gowning procedure before entering a Class B cleanroom?")
print("Operator: What's the gowning procedure before entering a Class B cleanroom?")
print("SOP Assistant:", answer_1, "\n")

answer_2 = await call_agent(sop_assistant_runner, "operator-1", "sop-session-1",
    "And how often does that gown need to be changed?")
print("Operator: And how often does that gown need to be changed?")
print("SOP Assistant:", answer_2)

---
# Example 3 — Batch Release Agent

**Use case:** A pharmacist asks "can this batch be released?" This
requires *acting*, not just answering — the agent must pull lab purity
data and check dosage math before it can respond. Unlike the LangGraph
version, there's no `ToolNode`/`tools_condition` wiring here — `LlmAgent`
runs the reason -> call tool -> observe -> reason loop internally.

In [ ]:
def check_batch_purity(batch_id: str) -> str:
    """Look up lab purity test results for a given batch ID."""
    lab_results = {
        "NM-2026-0091": "Purity: 99.6% | Impurity profile: within spec | Status: PASS",
        "NM-2026-0044": "Purity: 96.1% | Impurity profile: exceeds threshold | Status: FAIL",
    }
    return lab_results.get(batch_id, f"No lab record found for batch {batch_id}")

def calculate_dosage_variance(target_mg: float, measured_mg: float) -> str:
    """Calculate the % variance between target and measured dosage per unit."""
    variance_pct = abs(measured_mg - target_mg) / target_mg * 100
    verdict = "within USP tolerance" if variance_pct <= 5 else "OUT OF TOLERANCE"
    return f"Variance: {variance_pct:.2f}% - {verdict}"

def lookup_drug_monograph(drug_name: str) -> str:
    """Retrieve the regulatory monograph summary for a drug (dosage form, storage, indications)."""
    monographs = {
        "metformin": "Oral biguanide, 500-1000mg tablets, store below 25C, indicated for T2DM.",
    }
    return monographs.get(drug_name.lower(), f"No monograph found for {drug_name}")

batch_release_agent = Agent(
    name="batch_release_agent",
    model=get_model(),
    description="Decides whether a manufactured batch is cleared for release.",
    instruction="""You are the NovaMed Batch Release Agent. Before answering
    any release question, use your tools to check actual lab data — never
    guess purity or dosage figures.""",
    tools=[check_batch_purity, calculate_dosage_variance, lookup_drug_monograph],
)

batch_release_runner = Runner(
    agent=batch_release_agent, app_name=APP_NAME, session_service=novamed_session_service,
)

await new_session(user_id="pharmacist-1", session_id="release-session-1")
answer = await call_agent(batch_release_runner, "pharmacist-1", "release-session-1", (
    "Is batch NM-2026-0091 cleared for release? Check purity and confirm "
    "dosage variance for target 500mg, measured 512mg."
))
print(answer)

---
# Example 4 — Manufacturing Supervisor

**Use case:** A single agent juggling purity checks, regulatory lookups,
and document drafting gets confused about which tool applies. NovaOps
splits into specialists and adds a **coordinator** agent. In ADK, delegation
is automatic: the coordinator's `sub_agents` are described to its LLM, and
ADK's built-in transfer mechanism hands off the turn — no manual routing
model or conditional edges needed, unlike the LangGraph supervisor.

In [ ]:
quality_control_agent = Agent(
    name="quality_control_agent",
    model=get_model(),
    description="Verifies batch purity and dosage variance against spec.",
    instruction="""You are the Quality Control specialist. Always check the
    actual lab data via tools before giving a verdict.""",
    tools=[check_batch_purity, calculate_dosage_variance],
)

regulatory_agent = Agent(
    name="regulatory_agent",
    model=get_model(),
    description="Answers questions about drug monographs and compliance classification.",
    instruction="""You are the Regulatory Affairs specialist. Always consult
    the monograph tool before answering.""",
    tools=[lookup_drug_monograph],
)

documentation_agent = Agent(
    name="documentation_agent",
    model=get_model(),
    description="Drafts GMP-compliant batch records and summaries.",
    instruction="""You are the Documentation specialist. Draft clear,
    GMP-compliant batch records from information already gathered in the
    conversation.""",
)

manufacturing_supervisor_agent = Agent(
    name="manufacturing_supervisor_agent",
    model=get_model(),
    description="Coordinates NovaMed manufacturing specialists.",
    instruction="""You coordinate three specialists: quality_control_agent
    (purity/dosage), regulatory_agent (monographs/compliance), and
    documentation_agent (drafts batch records). Route each part of the
    pharmacist's question to the right specialist, then combine their
    answers into one final response.""",
    sub_agents=[quality_control_agent, regulatory_agent, documentation_agent],
)

manufacturing_supervisor_runner = Runner(
    agent=manufacturing_supervisor_agent, app_name=APP_NAME, session_service=novamed_session_service,
)

await new_session(user_id="pharmacist-1", session_id="supervisor-session-1")
answer = await call_agent(manufacturing_supervisor_runner, "pharmacist-1", "supervisor-session-1", (
    "Is batch NM-2026-0091 within dosage tolerance for target 500mg measured 512mg, "
    "and what's the monograph classification for metformin?"
))
print(answer)

---
# Example 5 — Formulation R&D Pipeline

**Use case:** Before NovaMed reformulates a drug, R&D needs a structured
literature review — plan sub-questions, search literature for each,
synthesize gaps, write a report. ADK's `SequentialAgent` runs a fixed
pipeline of agents in order; each step writes to session state via
`output_key`, and the next step reads it back using `{key}` templating in
its instruction — no `TypedDict`/`Annotated[list, add]` needed.

In [ ]:
from google.adk.agents import SequentialAgent

def search_pharma_literature(query: str) -> str:
    """Search pharmaceutical literature and patents for a formulation topic."""
    mock_results = {
        "bioavailability": "3 relevant papers on solubility enhancement via "
                            "nanocrystal formulation and lipid-based delivery.",
        "solubility": "2 papers on cyclodextrin complexation improving aqueous solubility.",
        "stability": "1 paper on polymorph screening to improve thermal stability.",
    }
    for keyword, result in mock_results.items():
        if keyword in query.lower():
            return result
    return f"Limited results for '{query}'."

formulation_planner_agent = Agent(
    name="formulation_planner_agent",
    model=get_model(),
    description="Breaks a formulation research question into subtasks.",
    instruction="""Break the research question into 3-4 short, specific
    subtasks, one per line, numbered.""",
    output_key="research_subtasks",
)

literature_searcher_agent = Agent(
    name="literature_searcher_agent",
    model=get_model(),
    description="Searches pharmaceutical literature for each research subtask.",
    instruction="""Subtasks to research: {research_subtasks}
    For each subtask, call search_pharma_literature and summarize the findings.""",
    tools=[search_pharma_literature],
    output_key="literature_findings",
)

gap_analyzer_agent = Agent(
    name="gap_analyzer_agent",
    model=get_model(),
    description="Synthesizes literature findings and identifies formulation gaps.",
    instruction="""Findings so far: {literature_findings}
    Synthesize these findings and identify open formulation gaps.""",
    output_key="gap_analysis",
)

rd_report_writer_agent = Agent(
    name="rd_report_writer_agent",
    model=get_model(),
    description="Writes the final formulation R&D report for the head chemist.",
    instruction="""Gap analysis: {gap_analysis}
    Write a short formulation R&D report with sections: Overview, Key
    Findings, Formulation Gaps, Recommended Next Steps.""",
    output_key="formulation_report",
)

formulation_rd_pipeline = SequentialAgent(
    name="formulation_rd_pipeline",
    description="Runs the full formulation R&D literature review pipeline in order.",
    sub_agents=[formulation_planner_agent, literature_searcher_agent,
                gap_analyzer_agent, rd_report_writer_agent],
)

formulation_rd_runner = Runner(
    agent=formulation_rd_pipeline, app_name=APP_NAME, session_service=novamed_session_service,
)

await new_session(user_id="chemist-1", session_id="rd-session-1")
final_report = await call_agent(formulation_rd_runner, "chemist-1", "rd-session-1",
    "How can we improve the bioavailability of Compound X?")
print(final_report)

---
# Example 6 — Drug Info & Pricing Assistant

**Use case:** Patients and pharmacists ask two kinds of questions once a
drug is released: "what does it treat" (needs retrieval-style lookup) and
"how much does it cost" (needs structured price lookup). Session-based
memory means a follow-up like "how much does it cost?" correctly resolves
to whichever drug was discussed earlier in the same session — no manual
`thread_id` wiring, ADK's `Runner` + `session_id` handles it.

In [ ]:
import pandas as pd

drug_price_table = pd.DataFrame([
    {"drug_name": "Metformin 500mg", "price_usd": 4.50},
    {"drug_name": "Amoxicillin 250mg", "price_usd": 6.20},
    {"drug_name": "Atorvastatin 20mg", "price_usd": 9.85},
])

drug_monograph_texts = {
    "metformin": "Metformin: oral biguanide for type 2 diabetes. Typical dose "
                 "500-1000mg twice daily with meals. Store below 25C. Common "
                 "side effect: GI upset.",
    "amoxicillin": "Amoxicillin: penicillin-class antibiotic. Typical dose "
                    "250-500mg every 8 hours. Avoid in penicillin allergy.",
    "atorvastatin": "Atorvastatin: statin for cholesterol management. Typical "
                     "dose 10-20mg once daily, evening administration preferred.",
}

def get_drug_price(drug_name: str) -> str:
    """Look up the retail price of a drug by name (substring match)."""
    matches = drug_price_table[
        drug_price_table["drug_name"].str.contains(drug_name, case=False, na=False)
    ]
    if matches.empty:
        return f"No pricing found for '{drug_name}'."
    row = matches.iloc[0]
    return f"{row['drug_name']}: ${row['price_usd']:.2f}"

def get_drug_info(drug_name: str) -> str:
    """Retrieve indications, dosage, and warnings for a drug from its monograph."""
    for key, text in drug_monograph_texts.items():
        if key in drug_name.lower():
            return text
    return f"No monograph found for '{drug_name}'."

drug_info_pricing_agent = Agent(
    name="drug_info_pricing_agent",
    model=get_model(),
    description="Answers patient and pharmacist questions about drug info and pricing.",
    instruction="""You answer questions about what a drug treats, its dosage,
    warnings, and its price. Use get_drug_info for monograph questions and
    get_drug_price for cost questions. If the patient refers to "it" or "that
    drug", infer which drug from earlier in this conversation.""",
    tools=[get_drug_price, get_drug_info],
)

drug_info_pricing_runner = Runner(
    agent=drug_info_pricing_agent, app_name=APP_NAME, session_service=novamed_session_service,
)

await new_session(user_id="patient-4471", session_id="drug-info-session-1")

r1 = await call_agent(drug_info_pricing_runner, "patient-4471", "drug-info-session-1",
    "What is Metformin used for and what's the typical dose?")
print("Patient: What is Metformin used for and what's the typical dose?")
print("Assistant:", r1, "\n")

r2 = await call_agent(drug_info_pricing_runner, "patient-4471", "drug-info-session-1",
    "How much does it cost?")
print("Patient: How much does it cost?")
print("Assistant:", r2)

---
# Example 7 — Raw Material Inventory Agent

**Use case:** Before formulation can start, the system needs to check and
decrement raw material stock. In the LangGraph version this needed a
manually built ReAct graph with explicit read/write routing. In ADK, a
plain `LlmAgent` with both a read tool and a write tool handles this with
no extra graph code — the framework's built-in tool loop covers it.

In [ ]:
raw_material_stock = pd.DataFrame([
    {"material_id": "RM-101", "quantity_kg": 250.0},
    {"material_id": "RM-204", "quantity_kg": 80.0},
])

def get_material_stock(material_id: str) -> str:
    """Query current stock level (kg) for a raw material by ID."""
    row = raw_material_stock[raw_material_stock["material_id"] == material_id]
    if row.empty:
        return f"No stock record for {material_id}"
    return f"{material_id}: {row.iloc[0]['quantity_kg']} kg available"

def consume_material(material_id: str, quantity_kg: float) -> str:
    """Deduct quantity_kg from a raw material's stock when a batch enters formulation."""
    idx = raw_material_stock.index[raw_material_stock["material_id"] == material_id]
    if len(idx) == 0:
        return f"No stock record for {material_id}"
    raw_material_stock.loc[idx, "quantity_kg"] -= quantity_kg
    remaining = raw_material_stock.loc[idx, "quantity_kg"].values[0]
    return f"Consumed {quantity_kg}kg of {material_id}. Remaining: {remaining}kg"

inventory_agent = Agent(
    name="inventory_agent",
    model=get_model(),
    description="Queries and updates NovaMed raw material stock levels.",
    instruction="""You manage raw material inventory. Use get_material_stock
    to check levels and consume_material to deduct stock when a batch enters
    formulation. Always confirm the resulting stock level after consuming.""",
    tools=[get_material_stock, consume_material],
)

inventory_runner = Runner(
    agent=inventory_agent, app_name=APP_NAME, session_service=novamed_session_service,
)

await new_session(user_id="operator-1", session_id="inventory-session-1")
answer = await call_agent(inventory_runner, "operator-1", "inventory-session-1",
    "How much RM-101 do we have, and consume 30kg of it for the next batch?")
print(answer)

---
# Example 8 — Batch Deviation Report Agent

**Use case:** When a batch fails QC, GMP requires a formal Deviation
Report with root cause and CAPA. ADK's `LoopAgent` runs a drafter and a
reviewer in a cycle; the reviewer calls an `exit_loop` tool once satisfied,
which sets `tool_context.actions.escalate = True` to break the loop —
replacing the LangGraph version's manual "check for the word APPROVED"
conditional edge with a purpose-built tool.

In [ ]:
from google.adk.agents import LoopAgent
from google.adk.tools.tool_context import ToolContext

def exit_loop(tool_context: ToolContext) -> dict:
    """Call this ONLY when the deviation report has been APPROVED by the QA
    reviewer, to end the drafting loop."""
    tool_context.actions.escalate = True
    return {"status": "loop_exited"}

deviation_drafter_agent = Agent(
    name="deviation_drafter_agent",
    model=get_model(),
    description="Drafts or revises the batch deviation report.",
    instruction="""Draft a GMP batch deviation report with sections: Incident
    Summary, Root Cause, Impact Assessment, CAPA (Corrective and Preventive
    Action). If review feedback exists in this conversation, revise the
    report to address it. Be specific and auditable.""",
    output_key="deviation_report_draft",
)

deviation_reviewer_agent = Agent(
    name="deviation_reviewer_agent",
    model=get_model(),
    description="Critiques the deviation report draft against GMP documentation standards.",
    instruction="""Current draft: {deviation_report_draft}
    Critique it: is the root cause specific enough? Is the CAPA actionable?
    If it meets GMP documentation standards, call exit_loop and say
    'APPROVED'. Otherwise, list concrete gaps for the drafter to fix.""",
    tools=[exit_loop],
)

deviation_report_loop = LoopAgent(
    name="deviation_report_loop",
    description="Drafts a batch deviation report through a draft-review-revise cycle.",
    sub_agents=[deviation_drafter_agent, deviation_reviewer_agent],
    max_iterations=3,
)

deviation_report_runner = Runner(
    agent=deviation_report_loop, app_name=APP_NAME, session_service=novamed_session_service,
)

await new_session(user_id="qa-manager-1", session_id="deviation-session-1")
result = await call_agent(deviation_report_runner, "qa-manager-1", "deviation-session-1", (
    "Batch NM-2026-0044 failed QC: purity 96.1%, impurity profile exceeded "
    "threshold. Root cause suspected: mixing time under-run by 12 minutes on "
    "Line 2. Draft the deviation report."
))
print(result)

---
# Example 9 — Plant Operations Router

**Use case:** A plant operator's chat interface shouldn't require them to
know which specialist to talk to. NovaOps composes the **Drug Info &
Pricing Agent** (Example 6) and the **Inventory Agent** (Example 7) as
`sub_agents` of one coordinator — reusing the exact agent objects built
earlier, not new copies. This is where earlier chapters stop being
separate demos and become one system.

In [ ]:
plant_operations_router_agent = Agent(
    name="plant_operations_router_agent",
    model=get_model(),
    description="Routes plant operator questions to the right NovaMed specialist.",
    instruction="""You route incoming questions:
    - Drug info or pricing questions -> drug_info_pricing_agent
    - Raw material stock questions -> inventory_agent
    - Greetings or small talk -> answer directly, briefly and warmly
    Delegate silently; don't announce which specialist you're using.""",
    sub_agents=[drug_info_pricing_agent, inventory_agent],
)

plant_operations_runner = Runner(
    agent=plant_operations_router_agent, app_name=APP_NAME, session_service=novamed_session_service,
)

await new_session(user_id="operator-2", session_id="plant-ops-session-1")
answer = await call_agent(plant_operations_runner, "operator-2", "plant-ops-session-1",
    "How much RM-204 do we have left?")
print(answer)

---
# Example 10 — Adverse Event Literature Screener

**Use case:** NovaMed's pharmacovigilance team screens case reports for a
safety review, applying an inclusion criterion that's easy to misapply on
a single pass (wrongly excluding pre-vetted regulatory-database reports).
This reuses the exact `LoopAgent` + `exit_loop` reflection pattern from
Example 8, applied per-report across a small in-memory batch.

In [ ]:
import re

INCLUSION_CRITERION = """
Exclude case reports lacking a causality assessment - EXCLUDING reports
sourced from recognized regulatory adverse-event databases (e.g., FAERS),
which are pre-vetted and should NOT be excluded on this basis.
"""

screener_agent = Agent(
    name="screener_agent",
    model=get_model(),
    description="Applies the inclusion criterion to a single adverse event case report.",
    instruction=f"""Criterion: {INCLUSION_CRITERION}
    If review feedback exists in this conversation, reconsider your decision.
    Respond with exactly: Decision: INCLUDE/EXCLUDE  Reason: <one sentence>""",
    output_key="screening_decision",
)

screening_reviewer_agent = Agent(
    name="screening_reviewer_agent",
    model=get_model(),
    description="Checks whether the screening decision correctly applied the criterion's exception.",
    instruction="""Decision under review: {screening_decision}
    Check: did the screener correctly avoid excluding a FAERS-sourced report
    just for lacking a causality assessment? If correct, call exit_loop and
    say 'APPROVED'. Otherwise explain the error for the screener to fix.""",
    tools=[exit_loop],
)

adverse_event_screening_loop = LoopAgent(
    name="adverse_event_screening_loop",
    description="Screens one adverse event case report through a screen-review-revise cycle.",
    sub_agents=[screener_agent, screening_reviewer_agent],
    max_iterations=3,
)

adverse_event_screener_runner = Runner(
    agent=adverse_event_screening_loop, app_name=APP_NAME, session_service=novamed_session_service,
)

sample_case_reports = pd.DataFrame([
    {"report_id": "AE-001", "source": "FAERS",
     "narrative": "Patient reported nausea after Metformin. No causality assessment on file."},
    {"report_id": "AE-002", "source": "Physician self-report",
     "narrative": "Patient reported dizziness. No causality assessment provided."},
])

async def screen_all_case_reports(case_reports: pd.DataFrame) -> pd.DataFrame:
    """Runs the screening loop across every case report and returns decisions as a DataFrame."""
    results = []
    for i, report in case_reports.iterrows():
        session_id = f"screening-session-{report['report_id']}"
        await new_session(user_id="pv-team", session_id=session_id)
        outcome = await call_agent(adverse_event_screener_runner, "pv-team", session_id,
            f"Source: {report['source']}\n{report['narrative']}")
        decision = re.search(r"Decision:\s*(INCLUDE|EXCLUDE)", outcome)
        reason = re.search(r"Reason:\s*(.+)", outcome)
        results.append({
            "report_id": report["report_id"],
            "decision": decision.group(1) if decision else "UNKNOWN",
            "reason": reason.group(1) if reason else outcome[:200],
        })
    return pd.DataFrame(results)

screening_results = await screen_all_case_reports(sample_case_reports)
screening_results

---
# Example 11 — Regulatory Guidance Summarizer

**Use case:** Regulatory affairs tracks dozens of FDA/EMA guidance
documents per year. Each needs a structured summary — scope, binding
requirements, what NovaOps must change to comply. This cell uses an
in-memory sample guidance text; the same tool-based PDF pattern from the
LangGraph guide (download -> extract -> summarize) applies here, just
wrapped as ADK tools instead of standalone functions.

In [ ]:
guidance_summarizer_agent = Agent(
    name="guidance_summarizer_agent",
    model=get_model(),
    description="Produces structured compliance summaries from regulatory guidance text.",
    instruction="""Summarize the regulatory guidance document you're given
    with exactly these sections: Scope, Key Requirements, Compliance
    Actions, Effective Date/Deadline.""",
)

guidance_summarizer_runner = Runner(
    agent=guidance_summarizer_agent, app_name=APP_NAME, session_service=novamed_session_service,
)

sample_guidance_text = """
FDA Guidance for Industry: Process Validation for Solid Oral Dosage Forms (2026 update).
This guidance applies to manufacturers of solid oral dosage forms (tablets, capsules).
Manufacturers must demonstrate process validation across three stages: process design,
process qualification, and continued process verification. Continuous monitoring data
must be retained for a minimum of 7 years. Manufacturers have 18 months from publication
to update their validation master plans. Effective date: January 1, 2027.
"""

await new_session(user_id="regulatory-team", session_id="guidance-session-1")
summary = await call_agent(guidance_summarizer_runner, "regulatory-team", "guidance-session-1",
    sample_guidance_text)
print(summary)

---
## Seeing the agent hierarchy

ADK doesn't have a direct equivalent to LangGraph's `draw_mermaid_png()` —
graphs of *tool calls and delegation* are runtime behavior, not a fixed
structure, since routing is decided by the LLM at each turn rather than
pre-declared edges. Two ways to inspect structure:

1. **Print the static `sub_agents` tree** (structure only, not runtime flow):

In [ ]:
def print_agent_tree(agent, indent=0):
    """Recursively prints an agent's sub_agents hierarchy."""
    print("  " * indent + f"- {agent.name}")
    for sub_agent in getattr(agent, "sub_agents", []) or []:
        print_agent_tree(sub_agent, indent + 1)

print_agent_tree(plant_operations_router_agent)

2. **Use `adk web`** from the terminal (`pip install google-adk` gives you
   this CLI) to get a live browser trace UI showing every delegation, tool
   call, and session state change as it actually happens — the closest ADK
   equivalent to watching a LangGraph execution.


---
## You've now built

The same 11-subsystem NovaOps platform as the LangGraph version — a batch
tracker, an SOP chatbot, a tool-using release agent, a coordinator of
specialists, a sequential R&D pipeline, a drug info assistant, an
inventory agent, two reflection loops, a router composing earlier agents,
and a document summarizer — this time on Google ADK, so you can directly
compare how the same 11 subsystems are expressed in each framework.
